# zkPHIRE: Programmable SumCheck for High-Degree Expressive Gates

**Based on:** *zkPHIRE: A Programmable Accelerator for ZKPs over HIgh-degRee, Expressive Gates*  
Daftardar et al., HPCA 2026

---

## What This Notebook Covers

| Section | Topic |
|---------|-------|
| 1 | Motivation: Limits of Fixed-Function SumCheck (zkSpeed) |
| 2 | High-Degree Composite Polynomial Structure |
| 3 | Extension Engine (EE): Computing MLE Extensions |
| 4 | Product Lane (PL): Multiplying Extensions |
| 5 | Graph-Based Scheduler for Data Reuse |
| 6 | Programmable SumCheck Implementation |
| 7 | MLE Update with Pipelining |
| 8 | ZeroCheck with Integrated f_r(X) |
| 9 | Benchmarks: Vanilla vs. Jellyfish Gates |
| 10 | Full zkPHIRE Performance Analysis |

---

## The Core Problem

**Notebook 1** showed how zkSpeed accelerates HyperPlonk's fixed *Vanilla* gate:  
$$f_{\text{plonk}} = q_L w_1 + q_R w_2 + q_M w_1 w_2 - q_O w_3 + q_C$$

Modern ZKP protocols use **expressive, high-degree gates**:  
- **Jellyfish** (HyperPlonk variant): $f = q_1 w_1 + q_2 w_2 + q_3 w_1^3 w_2 + q_4 w_1^4 + \ldots$ (degree 7)
- **Halo2**: custom gates for elliptic curve ops → up to degree 9  
- **Verifiable ASICs**: $f = q_{\text{add}}(a+b) + q_{\text{mul}}(a\cdot b)$ (different structure)

**zkSpeed's SumCheck unit cannot handle these** — it's hardwired for Vanilla gates.

**zkPHIRE** solves this with a *programmable* SumCheck unit that achieves **1486× speedup over CPU**.

In [ ]:
import random
import time
import operator
from functools import reduce
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict

random.seed(42)

# ---- Finite Field (same as Notebook 1) ----
class GF:
    def __init__(self, p):   self.p = p
    def add(self, a, b):     return (int(a) + int(b)) % self.p
    def sub(self, a, b):     return (int(a) - int(b)) % self.p
    def mul(self, a, b):     return (int(a) * int(b)) % self.p
    def neg(self, a):        return (-int(a)) % self.p
    def inv(self, a):        return pow(int(a), self.p - 2, self.p)
    def rand(self):          return random.randint(1, self.p - 1)
    def elem(self, x):       return int(x) % self.p

F       = GF(101)          # small prime for examples
F_bench = GF((1 << 31)-1)  # Mersenne prime for benchmarks

def make_random_mle(mu, F, seed=None):
    if seed is not None: random.seed(seed)
    return [F.rand() for _ in range(1 << mu)]

def mle_eval(table, point, F):
    mu = len(point)
    result = 0
    for i in range(1 << mu):
        chi = 1
        for j in range(mu):
            bit = (i >> (mu - 1 - j)) & 1
            chi = F.mul(chi, point[j] if bit else F.sub(1, point[j]))
        result = F.add(result, F.mul(table[i], chi))
    return result

def mle_update(table, r, F):
    n = len(table); half = n >> 1
    omr = F.sub(1, r)
    return [F.add(F.mul(omr, table[i]), F.mul(r, table[i+half])) for i in range(half)]

def sumcheck_prover_round(table, F):
    half = len(table) >> 1
    return sum(table[:half]) % F.p, sum(table[half:]) % F.p

print("Setup complete!  GF(101) and GF(2^31-1) ready.")

## Section 1: High-Degree Composite Polynomial Structure

A **composite polynomial** in HyperPlonk is:
$$f(X) = \sum_{t=1}^{T} \prod_{k \in \text{term}_t} \text{MLE}_k(X)$$

where each $\text{MLE}_k$ is a multilinear polynomial (degree 1 in each variable).  
A product of $d$ MLEs has **degree $d$** in each variable.

### Gate Type Comparison (from paper Table I)

| Gate Type | Polynomial | Terms | Max Degree |
|-----------|-----------|-------|------------|
| Vanilla Plonk | $q_L w_1 + q_R w_2 + q_M w_1 w_2 - q_O w_3 + q_C$ | 5 | 3 |
| Jellyfish ZeroCheck | $q_1 w_1 + q_2 w_2 + q_3 w_1^3 w_2 + q_4 w_1^4 + q_5 w_3 w_4 + q_{\text{ccc}} w_1 w_2 w_3 w_4$ | 6 | 7 |
| Jellyfish PermCheck | $(\pi - p_1 p_2 + \alpha (\phi D_1 D_2 D_3 - N_1 N_2 N_3 N_4))\cdot f$ | 2 | 9 |
| Verifiable ASICs | $(A+B-C)\cdot f_r$ | 1 | 3 |
| Halo2 ECC Add | $(1-x_p)\cdot \beta \cdot (x_r - x_p)\cdot (y_q + y_p)^2 + \ldots$ | 12 | 5 |

**Key challenge:** zkSpeed's SumCheck PE is optimized for Vanilla gates — it hardcodes the 5-term structure.  
Jellyfish gates have degree-7 terms and different composition — need *programmable* hardware.

In [ ]:
# ---- Polynomial Descriptor ----
# A composite polynomial is represented as a list of terms.
# Each term is a list of MLE table IDs to multiply together.
# Example: f = a*b*c + c*e + e*g  =>  terms = [['a','b','c'], ['c','e'], ['e','g']]

mu = 3

# Create named MLE tables (simulating HyperPlonk's MLEs)
def make_named_mles(names, mu, F, base_seed=0):
    """Create a dict of named MLE tables for a composite polynomial."""
    tables = {}
    for i, name in enumerate(names):
        tables[name] = make_random_mle(mu, F, seed=base_seed + i)
    return tables

# ---- Vanilla Plonk Gate (from zkSpeed paper) ----
vanilla_mle_names = ['q_L', 'q_R', 'q_M', 'q_O', 'q_C', 'w_1', 'w_2', 'w_3', 'f_z']
vanilla_tables    = make_named_mles(vanilla_mle_names, mu, F, base_seed=100)
vanilla_terms_ids = [
    ['q_L', 'w_1', 'f_z'],             # degree 3
    ['q_R', 'w_2', 'f_z'],             # degree 3
    ['q_M', 'w_1', 'w_2', 'f_z'],      # degree 4 (highest)
    ['q_C', 'f_z'],                    # degree 2
]

# ---- Jellyfish Gate (from zkPHIRE paper, f_zero) ----
jellyfish_mle_names = ['q_1','q_2','q_3','q_4','q_H1','q_H2','q_H3','q_H4',
                       'q_O','q_ccc','w_1','w_2','w_3','w_4','f_z']
jellyfish_tables    = make_named_mles(jellyfish_mle_names, mu, F, base_seed=200)
# Jellyfish ZeroCheck polynomial (from paper Table I, ID 22)
jellyfish_terms_ids = [
    ['q_1', 'w_1', 'f_z'],                                   # degree 3
    ['q_2', 'w_2', 'f_z'],                                   # degree 3
    ['q_3', 'w_1', 'w_1', 'w_1', 'w_2', 'f_z'],             # degree 6  (w_1^3 * w_2)
    ['q_4', 'w_1', 'w_1', 'w_1', 'w_1', 'f_z'],             # degree 6  (w_1^4)
    ['q_H1','w_1', 'f_z'],                                   # degree 3
    ['q_H2','w_2', 'f_z'],                                   # degree 3
    ['q_H3','w_3', 'f_z'],                                   # degree 3
    ['q_H4','w_4', 'f_z'],                                   # degree 3
    ['q_O', 'w_3', 'f_z'],                                   # degree 3
    ['q_ccc','w_1','w_2','w_3','w_4','f_z'],                 # degree 7 (highest!)
]

def poly_info(name, tables, term_ids):
    terms = [[tables[k] for k in tid] for tid in term_ids]
    degrees = [len(t) for t in terms]
    n_mle = len(set(k for tid in term_ids for k in tid))
    total_ops = sum(d * (1 << mu) // 2 for d in degrees)
    print(f"  {name}:")
    print(f"    Terms: {len(term_ids)},  Max degree: {max(degrees)},  Unique MLEs: {n_mle}")
    print(f"    Degrees per term: {degrees}")
    print(f"    Total EE extensions needed: ~{total_ops} for mu={mu}")
    return terms

print("Polynomial Analysis:")
vanilla_terms   = poly_info("Vanilla Plonk (zkSpeed)", vanilla_tables,   vanilla_terms_ids)
print()
jellyfish_terms = poly_info("Jellyfish Gate (zkPHIRE)", jellyfish_tables, jellyfish_terms_ids)
print()
print("Key insight: Jellyfish has 2.5x more terms, degree 7 vs 4.")
print("  zkSpeed's fixed SumCheck PE cannot handle degree-7 polynomials efficiently.")

## Section 3: Extension Engine (EE) — Computing MLE Extensions

The **Extension Engine** is the core compute unit in zkPHIRE's SumCheck.

### What it does
Given an MLE table entry pair $(v_0, v_1)$ at $(X_i=0, X_i=1)$:  
Compute the **linear extension** at integer evaluation points $t \in \{0, 1, \ldots, d\}$:

$$\text{ext}(t) = v_0 \cdot (1-t) + v_1 \cdot t$$

This is a **modular addition + modular multiplication** per evaluation point — exactly one modmul per extension.

### Why this is clever
For each entry in the boolean hypercube, we only need $(v_0, v_1)$ from the MLE table — not the full table.  
This allows **streaming** the MLE table through small on-chip scratchpads (tiles), dramatically reducing SRAM requirements compared to zkSpeed's large global scratchpad.

### Hardware (from Section III-B of paper)
- Each EE receives 2 (round 1) or 4 (later rounds) values from the MLE Update unit
- EE generates $d+1$ extensions per entry, fed directly to Product Lanes
- Pipelined: new entry every cycle after warmup

In [ ]:
class ExtensionEngine:
    """
    Extension Engine (EE) from zkPHIRE Section III-B.

    Computes linear extensions of MLE values to evaluation points 0..d.
    Each call simulates one 'slot' of EE computation for a single
    (MLE table, boolean hypercube entry) pair.

    Hardware: pipelined modular adder/subtracter chain.
    Cost per extension: 2 multiplications + 1 addition.
    """
    def __init__(self, F):
        self.F = F
        self.n_calls = 0
        self.n_muls  = 0

    def extend(self, v0, v1, eval_points):
        """
        Given MLE(X_i=0) = v0, MLE(X_i=1) = v1,
        compute MLE(X_i=t) for each t in eval_points.

        Formula: ext(t) = v0*(1-t) + v1*t  =  v0 + t*(v1-v0)
        """
        F = self.F
        self.n_calls += 1
        diff = F.sub(v1, v0)          # v1 - v0 (computed once)
        extensions = []
        for t in eval_points:
            if t == 0:
                ext = v0
            elif t == 1:
                ext = v1
            else:
                ext = F.add(v0, F.mul(t, diff))  # v0 + t*(v1-v0)
            extensions.append(ext)
            self.n_muls += (1 if t > 1 else 0)
        return extensions

# ---- Demo: EE for a 3-term product a*b*c ----
mu = 3
a = make_random_mle(mu, F, seed=1)
b = make_random_mle(mu, F, seed=2)
c = make_random_mle(mu, F, seed=3)

ee = ExtensionEngine(F)
eval_pts = [0, 1, 2, 3]   # need d+1=4 points for degree-3 product

print("Extension Engine Demo: MLE tables a, b, c")
print(f"Computing extensions for entry=0 (x2=0, x3=0) across eval points {eval_pts}:")
print()

entry = 0
half  = 1 << (mu - 1)   # 4

for name, tbl in [('a', a), ('b', b), ('c', c)]:
    v0 = tbl[entry]
    v1 = tbl[entry + half]
    exts = ee.extend(v0, v1, eval_pts)
    print(f"  MLE {name}:  v0={v0}, v1={v1}")
    for t, ext in zip(eval_pts, exts):
        formula = f"{v0}*(1-{t}) + {v1}*{t} = {ext}"
        print(f"    ext(X_i={t}) = {formula}")
    print()

# Verify: extension at t=0,1 should match original table values
print(f"Verification for MLE a at t=0: ext={ee.extend(a[entry], a[entry+half], [0])[0]} == a[entry]={a[entry]}  {'✓' if ee.extend(a[entry], a[entry+half], [0])[0] == a[entry] else '✗'}")
print(f"Verification for MLE a at t=1: ext={ee.extend(a[entry], a[entry+half], [1])[0]} == a[entry+half]={a[entry+half]}  {'✓' if ee.extend(a[entry], a[entry+half], [1])[0] == a[entry+half] else '✗'}")

## Section 4: Product Lane (PL) — Multiplying Extensions

The **Product Lane** multiplies extensions from multiple EEs for a single product term.

### Datapath (Figure 3 in paper)
```
MLE_1 → [EE] → ext_1(0..d) ─┐
MLE_2 → [EE] → ext_2(0..d) ─┤──→ [PL] → product(0..d) ──→ [Accumulator]
MLE_k → [EE] → ext_k(0..d) ─┘
```

For **d+1** evaluation points, the PL computes **d+1 products** in parallel (one per register).

### Data Reuse via Graph Decomposition (Figure 2 in paper)
For $f = a \cdot b \cdot c \cdot e \cdot g + h + h \cdot k \cdot n + a \cdot b \cdot c \cdot e \cdot g + h + h \cdot k \cdot n$:
- **Balanced tree** schedule: requires 2 temporary MLE buffers
- **Accumulation-based** schedule: requires only 1 Tmp MLE buffer

The scheduler decides which MLE tables to route to EEs at each step, minimizing temporary storage while keeping Product Lanes busy.

In [ ]:
class ProductLane:
    """
    Product Lane (PL) from zkPHIRE Section III-B.

    Maintains an accumulator that holds the running product of extensions
    across all MLE factors in the current term.
    One PL handles all (d+1) evaluation points simultaneously.

    Hardware: E-1 pipelined modular multipliers (E = number of evaluation points).
    """
    def __init__(self, F, n_eval_pts):
        self.F          = F
        self.n_eval_pts = n_eval_pts
        self.acc        = None      # running product: list of n_eval_pts values
        self.n_muls     = 0

    def reset(self):
        """Start a new product term."""
        self.acc = [1] * self.n_eval_pts

    def multiply(self, extensions):
        """
        Multiply a new set of MLE extensions into the running product.
        extensions[i] = MLE(X_i = eval_pt_i) for all eval points.
        """
        assert len(extensions) == self.n_eval_pts
        if self.acc is None:
            self.reset()
        for i in range(self.n_eval_pts):
            self.acc[i] = self.F.mul(self.acc[i], extensions[i])
            self.n_muls += 1

    def get_product(self):
        """Return current product accumulator."""
        return self.acc[:] if self.acc else [0] * self.n_eval_pts

# ---- Demo: PL for f = a*b*c  at entry=0 ----
degree    = 3      # product of 3 MLEs
eval_pts  = list(range(degree + 1))   # [0,1,2,3]
ee        = ExtensionEngine(F)
pl        = ProductLane(F, len(eval_pts))

print(f"Product Lane Demo: f = a*b*c  at entry=0, eval_pts={eval_pts}")
print()

pl.reset()
for name, tbl in [('a', a), ('b', b), ('c', c)]:
    v0   = tbl[entry]
    v1   = tbl[entry + half]
    exts = ee.extend(v0, v1, eval_pts)
    pl.multiply(exts)
    print(f"  After multiplying {name}:  product = {pl.get_product()}")

product_at_entry = pl.get_product()
print()

# Verify: product at t=0 should be a[0]*b[0]*c[0]
expected_t0 = F.mul(F.mul(a[0], b[0]), c[0])
expected_t1 = F.mul(F.mul(a[half], b[half]), c[half])
print(f"  Verify t=0: PL={product_at_entry[0]} == a[0]*b[0]*c[0]={expected_t0}  {'✓' if product_at_entry[0]==expected_t0 else '✗'}")
print(f"  Verify t=1: PL={product_at_entry[1]} == a[half]*b[half]*c[half]={expected_t1}  {'✓' if product_at_entry[1]==expected_t1 else '✗'}")

## Section 5: Graph-Based Scheduler

The **scheduler** determines the order in which MLE tables are processed by EEs and PLs.  
This is a core contribution of zkPHIRE (Section III-C, Figure 2).

### The Scheduling Problem
For $f = a \cdot b \cdot c \cdot e + c \cdot e + e \cdot g$ (example from the paper):
- 3 terms share MLEs $c$, $e$
- Naively processing each term independently re-computes $c$ and $e$ extensions
- Optimal schedule reuses $c$ and $e$ across terms

### Accumulation-Based Schedule
Instead of a balanced binary tree (requires 2 Tmp buffers), zkPHIRE uses an **accumulation-based** schedule:
- Process terms one by one, accumulating the running product into a single Tmp MLE buffer
- Only **1 Tmp buffer** needed (vs 2 for tree approach)
- Same number of total steps as balanced tree

### Scheduler Output
The scheduler emits a list of `(step, mle_id, ee_id, pl_id, is_term_start, is_term_end)` operations  
that guide the SumCheck hardware through all terms of the composite polynomial.

In [ ]:
def build_accumulation_schedule(term_ids, all_tables):
    """
    Build an accumulation-based computation schedule for a composite polynomial.

    term_ids:   list of terms, each term = list of MLE names (strings)
    all_tables: dict mapping MLE name -> table

    Returns a list of ScheduleStep namedtuples specifying the EE/PL datapath.
    This simulates zkPHIRE's automated scheduler (Section III-E).

    Key optimization: MLEs appearing in multiple terms are marked for prefetch.
    """
    # Count how many times each MLE appears across terms (for reuse analysis)
    mle_count = defaultdict(int)
    for term in term_ids:
        for mle_name in term:
            mle_count[mle_name] += 1

    schedule = []
    step_id  = 0
    for t_idx, term in enumerate(term_ids):
        for k_idx, mle_name in enumerate(term):
            # Determine if next MLE should be prefetched (data reuse optimization)
            next_term = term_ids[t_idx + 1] if t_idx + 1 < len(term_ids) else []
            is_shared  = mle_count[mle_name] > 1
            prefetch   = (k_idx == len(term) - 2) and len(next_term) > 0

            schedule.append({
                'step':          step_id,
                'term_idx':      t_idx,
                'mle_idx':       k_idx,
                'mle_name':      mle_name,
                'table':         all_tables[mle_name],
                'is_term_start': k_idx == 0,
                'is_term_end':   k_idx == len(term) - 1,
                'is_shared':     is_shared,
                'prefetch_next': prefetch,
            })
            step_id += 1
    return schedule, mle_count

# Build schedule for Jellyfish ZeroCheck
schedule, mle_count = build_accumulation_schedule(jellyfish_terms_ids, jellyfish_tables)

print("Schedule for Jellyfish ZeroCheck:")
print(f"  Total steps: {len(schedule)}")
print()
print("MLE reuse analysis:")
shared_mles = {k: v for k, v in mle_count.items() if v > 1}
for mle, count in sorted(shared_mles.items(), key=lambda x: -x[1]):
    print(f"  {mle}: appears in {count} terms  →  prefetch and reuse!")
print()
print("First 10 schedule steps:")
print(f"{'Step':>5} {'Term':>5} {'MLE':>10} {'Start?':>8} {'End?':>6} {'Shared?':>8}")
print("-" * 48)
for step in schedule[:10]:
    print(f"{step['step']:>5} {step['term_idx']:>5} {step['mle_name']:>10} "
          f"{'Yes' if step['is_term_start'] else '':>8} "
          f"{'Yes' if step['is_term_end'] else '':>6} "
          f"{'Yes' if step['is_shared'] else '':>8}")

## Section 6: Programmable SumCheck — Full Implementation

Putting it all together: the complete programmable SumCheck unit from zkPHIRE.

### Datapath for one SumCheck round
```
For each entry in boolean hypercube (0..half-1):
  For each step in schedule:
    v0, v1 = MLE_table[entry], MLE_table[entry + half]
    exts = EE.extend(v0, v1, eval_points)       ← Extension Engine
    if is_term_start: PL.reset()
    PL.multiply(exts)                            ← Product Lane
    if is_term_end: accumulator += PL.get_product() ← Accumulator

Return g_evals = accumulator  (g(0), g(1), ..., g(d))
```

### For subsequent rounds
After each SumCheck round, all MLE tables are updated with the verifier's challenge.  
The schedule structure stays the same — only the table contents change.

In [ ]:
def programmable_sumcheck_round(term_ids, all_tables, F, verbose=False):
    """
    One round of programmable SumCheck over a composite polynomial.

    Implements the full EE + PL + Accumulator datapath from zkPHIRE Figure 3.
    Returns g_evals: list of g(0), g(1), ..., g(max_degree).
    """
    max_degree = max(len(term) for term in term_ids)
    eval_pts   = list(range(max_degree + 1))
    n_eval_pts = len(eval_pts)
    half       = len(list(all_tables.values())[0]) >> 1

    ee   = ExtensionEngine(F)
    pl   = ProductLane(F, n_eval_pts)
    g_acc = [0] * n_eval_pts   # final accumulator: g(0)..g(max_degree)

    for entry in range(half):
        pl.reset()
        current_term = -1
        entry_term_sum = [0] * n_eval_pts

        for t_idx, term in enumerate(term_ids):
            pl.reset()
            for mle_name in term:
                tbl  = all_tables[mle_name]
                v0   = tbl[entry]
                v1   = tbl[entry + half]
                exts = ee.extend(v0, v1, eval_pts)    # EE
                pl.multiply(exts)                     # PL
            prods = pl.get_product()
            for i in range(n_eval_pts):
                entry_term_sum[i] = F.add(entry_term_sum[i], prods[i])

        for i in range(n_eval_pts):
            g_acc[i] = F.add(g_acc[i], entry_term_sum[i])

    if verbose:
        print(f"  EE calls: {ee.n_calls}, PL muls: {pl.n_muls}")
    return g_acc, eval_pts

# Verify correctness on a simple case
def direct_poly_sum(term_ids, tables, F, mu):
    """Brute-force: compute sum_{x in {0,1}^mu} f(x) directly."""
    total = 0
    for i in range(1 << mu):
        f_val = 0
        for term in term_ids:
            prod = 1
            for mle_name in term:
                prod = F.mul(prod, tables[mle_name][i])
            f_val = F.add(f_val, prod)
        total = F.add(total, f_val)
    return total

# Test with vanilla polynomial
random.seed(42)
H_vanilla_direct = direct_poly_sum(vanilla_terms_ids, vanilla_tables, F, mu)
g_vanilla, eps_v  = programmable_sumcheck_round(vanilla_terms_ids, vanilla_tables, F)
check_v = F.add(g_vanilla[0], g_vanilla[1])
print(f"Vanilla SumCheck Round 1:")
print(f"  H (direct) = {H_vanilla_direct}")
print(f"  g(0)+g(1)  = {check_v}")
print(f"  Consistency: {'✓' if check_v == H_vanilla_direct else '✗'}")

print()

# Test with Jellyfish polynomial
H_jelly_direct = direct_poly_sum(jellyfish_terms_ids, jellyfish_tables, F, mu)
g_jelly, eps_j  = programmable_sumcheck_round(jellyfish_terms_ids, jellyfish_tables, F)
check_j = F.add(g_jelly[0], g_jelly[1])
print(f"Jellyfish SumCheck Round 1:")
print(f"  H (direct)  = {H_jelly_direct}")
print(f"  g(0)+g(1)   = {check_j}")
print(f"  Max degree:   {len(g_jelly)-1} (evaluations at {eps_j})")
print(f"  Consistency:  {'✓' if check_j == H_jelly_direct else '✗'}")

## Section 7: Full Multi-Round Programmable SumCheck

Run the complete μ-round protocol, updating all MLE tables after each round.

In [ ]:
def lagrange_eval_from_0(evals, x, F):
    """
    Evaluate degree-(n-1) polynomial at x, given evals at 0,1,...,n-1.
    Lagrange interpolation — hardware uses Barycentric form for efficiency.
    """
    n = len(evals)
    result = 0
    for i in range(n):
        num = evals[i]
        for j in range(n):
            if j != i:
                num = F.mul(num, F.mul(F.sub(x, j), F.inv(F.sub(i, j))))
        result = F.add(result, num)
    return result

def run_programmable_sumcheck(term_ids, tables_init, mu, F, verbose=True):
    """
    Full programmable SumCheck protocol for a composite polynomial.

    Runs mu rounds, each with:
    1. Prover computes g_i(0..d) via EE + PL datapath
    2. Verifier checks consistency, sends random challenge r_i
    3. All MLE tables updated: T[k] <- mle_update(T[k], r_i)
    """
    # Copy tables so we don't modify originals
    tables     = {k: list(v) for k, v in tables_init.items()}
    H          = direct_poly_sum(term_ids, tables, F, mu)
    transcript = []
    challenges = []
    prev_claim = H

    if verbose:
        print(f"{'='*65}")
        print(f"Programmable SumCheck  |  mu={mu}  |  H={H}")
        max_deg = max(len(t) for t in term_ids)
        print(f"Terms={len(term_ids)}, Max_degree={max_deg}, Unique_MLEs={len(tables)}")
        print(f"{'='*65}")

    for round_i in range(mu):
        # PROVER: EE + PL computation
        g_evals, eval_pts = programmable_sumcheck_round(term_ids, tables, F)

        # VERIFIER: consistency check
        check = F.add(g_evals[0], g_evals[1])
        assert check == prev_claim, f"Round {round_i+1} FAILED: {check} != {prev_claim}"

        # VERIFIER: random challenge
        r = F.rand()
        challenges.append(r)
        transcript.append(g_evals[:])

        if verbose:
            print(f"  Round {round_i+1}: g(0)={g_evals[0]:3d} g(1)={g_evals[1]:3d} "
                  f"sum={check:3d}=prev ✓  r{round_i+1}={r}")

        # Update claim using Lagrange eval at r
        prev_claim = lagrange_eval_from_0(g_evals, r, F)

        # All MLE tables updated with challenge r
        for k in tables:
            tables[k] = mle_update(tables[k], r, F)

    # Final claim
    final_claim = prev_claim

    if verbose:
        print(f"  Final claim: {final_claim}")
        print(f"  SumCheck VERIFIED in {mu} rounds  ✅")

    return H, transcript, challenges, final_claim

# Run for Jellyfish gates
random.seed(42)
H, transcript, challenges, final = run_programmable_sumcheck(
    jellyfish_terms_ids, jellyfish_tables, mu, F
)

## Section 8: ZeroCheck with Integrated f_r(X)

In zkPHIRE's ZeroCheck, the $f_r(X) = \text{eq}(\alpha, X)$ polynomial is computed **on-the-fly** during the first SumCheck round, rather than pre-computing the full table.

### Why? (Section III-F in paper)
- Pre-computing $f_r$ requires O(2^μ) writes and reads — extra bandwidth
- Building $f_r$ on-the-fly during the first round avoids this memory overhead
- One Product Lane is dedicated to computing $f_r(X)$, while remaining PLs handle the actual polynomial terms
- This saves $O(N)$ writes + reads of a 255-bit MLE table — significant at 2^{24}+ problem sizes

### Implementation
During round 1, for each entry $i$:
- The dedicated PL computes $f_r(i) = \prod_j (\alpha_j \cdot x_j + (1-\alpha_j)\cdot(1-x_j))$
- This value multiplies each term's product at that entry
- No separate table write needed!

In [ ]:
def compute_eq_entry(alpha, entry_idx, mu, F):
    """
    Compute eq(alpha, x) for a specific boolean entry x = binary_decode(entry_idx).
    This is what the 'Build MLE on-the-fly' does for each entry during round 1.
    """
    val = 1
    for j in range(mu):
        bit    = (entry_idx >> (mu - 1 - j)) & 1
        factor = alpha[j] if bit else F.sub(1, alpha[j])
        val    = F.mul(val, factor)
    return val

def zerocheck_with_integrated_fr(term_ids, tables_init, mu, F, verbose=True):
    """
    ZeroCheck with f_r(X) computed on-the-fly during round 1.

    Round 1 is special: for each entry, the product of all terms is
    multiplied by eq(alpha, x) before accumulation.

    Subsequent rounds run standard programmable SumCheck on the
    updated tables (eq is already folded in via the MLE Update).
    """
    # Verifier sends ZeroCheck challenges alpha
    alpha    = [F.rand() for _ in range(mu)]
    tables   = {k: list(v) for k, v in tables_init.items()}
    n_pts    = max(len(t) for t in term_ids) + 2  # +1 for eq factor
    eval_pts = list(range(n_pts))
    half     = len(list(tables.values())[0]) >> 1

    if verbose:
        print(f"ZeroCheck with on-the-fly f_r(X)")
        print(f"  alpha = {alpha}")
        print(f"  ZeroCheck claim: sum_x f(x)*eq(alpha,x) should = 0 if f is the zero polynomial")

    # Round 1: integrate eq(alpha, x) on-the-fly
    ee    = ExtensionEngine(F)
    pl    = ProductLane(F, n_pts)
    g_acc = [0] * n_pts

    for entry in range(half):
        entry_sum = [0] * n_pts

        # eq factor at this entry (Boolean, just the scalar)
        eq_v0 = compute_eq_entry(alpha, entry,      mu, F)
        eq_v1 = compute_eq_entry(alpha, entry+half, mu, F)
        eq_exts = ee.extend(eq_v0, eq_v1, eval_pts)

        for term in term_ids:
            pl.reset()
            # Multiply eq extensions first
            pl.multiply(eq_exts)
            # Then multiply all MLE factors
            for mle_name in term:
                tbl  = tables[mle_name]
                v0   = tbl[entry]
                v1   = tbl[entry + half]
                exts = ee.extend(v0, v1, eval_pts)
                pl.multiply(exts)
            prods = pl.get_product()
            for i in range(n_pts):
                entry_sum[i] = F.add(entry_sum[i], prods[i])

        for i in range(n_pts):
            g_acc[i] = F.add(g_acc[i], entry_sum[i])

    H_check = F.add(g_acc[0], g_acc[1])

    if verbose:
        print(f"  Round 1 sum (should be 0 for valid circuit): {H_check}")
        print(f"  g(0)={g_acc[0]}, g(1)={g_acc[1]}, g(0)+g(1)={H_check}")

    return H_check, g_acc

# Test: f should NOT be zero (random tables)
random.seed(42)
print("Test 1: Random polynomial (should NOT be zero)")
H_check, _ = zerocheck_with_integrated_fr(vanilla_terms_ids, vanilla_tables, mu, F)
print(f"  H = {H_check}  ({'Zero (false positive!)' if H_check == 0 else 'Non-zero as expected ✓'})")

print()
print("Test 2: Zero polynomial (all tables = 0)")
zero_tables = {k: [0]*(1<<mu) for k in vanilla_tables}
H_check2, _ = zerocheck_with_integrated_fr(vanilla_terms_ids, zero_tables, mu, F)
print(f"  H = {H_check2}  ({'Zero as expected ✓' if H_check2 == 0 else 'Non-zero (bug!)'})")

## Section 9: Benchmarks — Vanilla vs. Jellyfish Gates

The key result of zkPHIRE: using Jellyfish gates **reduces the gate count** by up to 32× for some circuits  
(hashing, ECC), while zkPHIRE handles the more complex SumCheck with only a modest slowdown per gate.

**Net result:** Jellyfish + zkPHIRE often wins over Vanilla + zkSpeed end-to-end (Table VII in paper).

Let's measure the per-round SumCheck cost for each gate type.

In [ ]:
def benchmark_sumcheck_round(term_ids_list, table_configs, mu_vals, n_trials=2):
    """
    Benchmark one SumCheck round for different polynomial structures.
    Measures cost in terms of:
    - Wall-clock time (Python simulation)
    - Number of modular multiplications (hardware cost)
    """
    results = []
    for name, term_ids in term_ids_list:
        for mu in mu_vals:
            tables = {k: make_random_mle(mu, F_bench, seed=hash(k)%1000)
                      for k in set(mle for t in term_ids for mle in t)}

            t0 = time.perf_counter()
            for _ in range(n_trials):
                g_evals, _ = programmable_sumcheck_round(term_ids, tables, F_bench)
            elapsed_ms = (time.perf_counter() - t0) / n_trials * 1000

            # Count theoretical modmuls
            max_deg = max(len(t) for t in term_ids)
            n_eval  = max_deg + 1
            n_entry = 1 << (mu - 1)
            # Per entry per term: (degree * n_eval) multiplications in PL
            total_muls = sum(len(t) * n_eval for t in term_ids) * n_entry

            results.append({
                'name': name, 'mu': mu, 'n': 1<<mu,
                'time_ms': elapsed_ms, 'n_muls': total_muls,
                'max_degree': max_deg, 'n_terms': len(term_ids)
            })

    return results

mu_vals = [6, 8, 10]

# Define compact polynomial descriptions for benchmarking
# (use small mu so runtime is reasonable in Python)
bench_polys = [
    ('Vanilla (d=4)', vanilla_terms_ids),
    ('Jellyfish ZeroCheck (d=7)', jellyfish_terms_ids),
]

print("Benchmarking one SumCheck round (Python simulation):")
results = benchmark_sumcheck_round(bench_polys, None, mu_vals)

print(f"{'Name':<30} {'mu':>4} {'n':>8} {'Time (ms)':>12} {'Muls (est)':>12}")
print("-" * 70)
for r in results:
    print(f"{r['name']:<30} {r['mu']:>4} {r['n']:>8} {r['time_ms']:>12.3f} {r['n_muls']:>12,}")

# Overhead ratio
print()
print("Jellyfish overhead vs Vanilla (per-round):")
for mu in mu_vals:
    v_time = next(r['time_ms'] for r in results if r['name']=='Vanilla (d=4)'         and r['mu']==mu)
    j_time = next(r['time_ms'] for r in results if r['name']=='Jellyfish ZeroCheck (d=7)' and r['mu']==mu)
    print(f"  mu={mu}: Jellyfish/Vanilla = {j_time/v_time:.2f}x  (degree-7 vs degree-4)")

In [ ]:
# ---- System-Level Analysis (from paper data) ----
# zkPHIRE paper Table VII: runtime comparison at iso-CPU area

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ---- Plot 1: Speedups over CPU for different gate types ----
ax = axes[0]
workloads = ['ZCash', 'Zexe\nCircuit', 'Rollup\n10 Pvt Tx', '2^12\nRescue Hash', 'Rollup\n25 Pvt Tx']
zkspeed_speedup    = [720, 862, 859, 844, None]   # from Table 3, zkSpeed paper
zkphire_v_speedup  = [783, 969, 972, 888, 957]    # from Table VI, zkPHIRE (Vanilla)
zkphire_j_speedup  = [934, 1354, 1471, None, 1590] # from Table VII, zkPHIRE (Jellyfish)

x = np.arange(len(workloads))
width = 0.28

# Filter None values
def safe_bar(ax, x, values, **kwargs):
    vals = [v if v is not None else 0 for v in values]
    bars = ax.bar(x, vals, **kwargs)
    for i, (bar, v) in enumerate(zip(bars, values)):
        if v is not None:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                    f'{v}×', ha='center', va='bottom', fontsize=8)
    return bars

safe_bar(ax, x - width, zkspeed_speedup,   width=width, label='zkSpeed (Vanilla)',   color='#2196F3', alpha=0.85)
safe_bar(ax, x,         zkphire_v_speedup, width=width, label='zkPHIRE (Vanilla)',   color='#4CAF50', alpha=0.85)
safe_bar(ax, x + width, zkphire_j_speedup, width=width, label='zkPHIRE (Jellyfish)', color='#FF5722', alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(workloads, fontsize=9)
ax.set_ylabel('Speedup over CPU', fontsize=11)
ax.set_title('Speedup Comparison at Iso-CPU Area\n(from paper Tables III, VI, VII)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# ---- Plot 2: SumCheck runtime breakdown across polynomial degrees ----
ax2 = axes[1]
degrees    = list(range(2, 15))
# Cost model: O(d * n) where d = max degree, n = 2^mu
n          = 2**20  # representative
vanilla_cost = 4 * n  # fixed degree 4
jelly_cost   = [d * n for d in degrees]

ax2.plot(degrees, [c / vanilla_cost for c in jelly_cost], 'r-o', label='Relative SumCheck cost', markersize=6)
ax2.axhline(y=1.0, color='blue', linestyle='--', label='Vanilla baseline (d=4)')
ax2.axvline(x=7,   color='green', linestyle=':', linewidth=2, label='Jellyfish degree=7')
ax2.fill_between(degrees, [c/vanilla_cost for c in jelly_cost], 1.0,
                 where=[c/vanilla_cost > 1.0 for c in jelly_cost],
                 alpha=0.15, color='red', label='Extra SumCheck cost')
ax2.set_xlabel('Polynomial Degree (d)', fontsize=11)
ax2.set_ylabel('Relative SumCheck Cost', fontsize=11)
ax2.set_title('SumCheck Cost vs Polynomial Degree\n(normalized to Vanilla d=4)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ---- Plot 3: zkPHIRE Area Breakdown (from paper Table V) ----
ax3 = axes[2]
modules_area  = ['MSM\n(32 PEs)', 'Multifunction\nForest (80T)', 'SumCheck\n(16 PEs)',
                 'Other', 'SRAM', 'HBM3\n(2 PHYs)', 'Interconnect']
compute_area  = [105.69, 48.18, 16.65, 10.64, 27.55, 59.20, 26.42]
colors_pie    = ['#2196F3', '#FF9800', '#4CAF50', '#9E9E9E', '#F44336', '#9C27B0', '#00BCD4']

wedges, texts, autotexts = ax3.pie(
    compute_area, labels=modules_area, autopct='%1.1f%%',
    colors=colors_pie, pctdistance=0.78, startangle=90
)
for text in texts:     text.set_fontsize(8)
for at in autotexts:   at.set_fontsize(7)
ax3.set_title(f'zkPHIRE Area Breakdown\nTotal: 294 mm²  |  Power: 202 W', fontsize=11)

plt.tight_layout()
plt.savefig('zkphire_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: zkphire_analysis.png")

In [ ]:
# ---- Gate Count Reduction Analysis (Table VII from paper) ----
# Jellyfish reduces gate count vs Vanilla, and zkPHIRE handles the resulting SumChecks

print("Gate Count Reduction with Jellyfish Gates (Table VII from paper):")
print(f"{'Workload':<25} {'Vanilla Gates':>15} {'Jellyfish Gates':>16} {'Reduction':>10}")
print("-" * 70)

workloads_gates = [
    ('ZCash',              2**17, 2**15, '4x'),
    ('Zexe Recursive Ckt', 2**22, 2**17, '32x'),
    ('Rollup 10 Pvt Tx',   2**23, 2**18, '32x'),
    ('2^12 Rescue Hashes', 2**21, 2**20, '2x'),
    ('Rollup 25 Pvt Tx',   2**24, 2**19, '32x'),
]
for name, v_gates, j_gates, red in workloads_gates:
    print(f"  {name:<23} {v_gates:>15,} {j_gates:>16,} {red:>10}")

print()
print("System runtime with zkPHIRE (Table VII):")
print(f"{'Workload':<25} {'zkSpeed+ Vanilla':>18} {'zkPHIRE Jellyfish':>18} {'Speedup':>8}")
print("-" * 73)

workloads_runtime = [
    ('ZCash',              1.825,  0.750,  '2.43x'),
    ('Zexe Recursive Ckt', 38.535, 1.440,  '26.76x'),
    ('Rollup 10 Pvt Tx',   76.356, 2.269,  '33.65x'),
    ('2^12 Rescue Hashes', 19.631, 7.114,  '2.76x'),
    ('Rollup 25 Pvt Tx',   151.97, 3.874,  '39.23x'),
]
for name, zks, zkp, sp in workloads_runtime:
    print(f"  {name:<23} {zks:>16.3f}ms {zkp:>16.3f}ms {sp:>8}")

print()
print("Key insight: gate reduction × programmable SumCheck = massive end-to-end speedup!")
print("  For workloads with 32x gate reduction, total speedup can exceed 30x over zkSpeed.")

## Summary: zkPHIRE Architecture Takeaways

### What We Implemented

| Algorithm | Paper Section | Key Insight |
|-----------|--------------|-------------|
| `ExtensionEngine` | §III-B | Linear extend v0,v1 to 0..d — 1 modmul/point |
| `ProductLane` | §III-B | Accumulate products across MLE factors — pipelined |
| `build_accumulation_schedule` | §III-C | Graph decomposition minimizes tmp buffers |
| `programmable_sumcheck_round` | §III | EE + PL + Acc for arbitrary composite polys |
| `run_programmable_sumcheck` | §III | Full multi-round protocol with MLE updates |
| `zerocheck_with_integrated_fr` | §III-F | On-the-fly eq(α,X) saves O(N) bandwidth |

### zkPHIRE vs zkSpeed: Key Differences

| Aspect | zkSpeed | zkPHIRE |
|--------|---------|----------|
| Gate support | Vanilla Plonk only | Any composite polynomial |
| Max degree | 4 | 30+ |
| SumCheck unit | Fixed-function PE | Programmable EE + PL |
| SRAM | Large global scratchpad | Small per-tile scratchpad |
| MSM overhead | Included | Same (uses zkSpeed's MSM) |
| Perf vs zkSpeed+ | Baseline | ~10% slower (per gate), but gates↓ |
| Jellyfish speedup | N/A | Up to **1486×** over CPU |

### Design Trade-off: Programmability vs Efficiency

```
                      zkSpeed (fixed)    zkPHIRE (programmable)
SumCheck unit area:      30.8 mm²             35.2 mm² (+14%)
SRAM:                  ~300 MB (large)         ~4.9 MB (tiles)
Gate types:            Vanilla only            Any degree
Jellyfish support:      ✗                       ✓
Scales to 2^30:         ✗ (SRAM grows)          ✓ (fixed tile size)
```

### The Big Picture
1. **SumCheck** over composite polynomials is O(d · n) per round — programmability adds flexibility at modest cost
2. **Extension Engines** are the key hardware primitive — one modmul per (MLE, eval point) pair
3. **Product Lanes** replace the hardwired multiply trees in zkSpeed — now support arbitrary term counts
4. **Graph scheduling** minimizes data movement — the scheduler is the 'compiler' for the SumCheck hardware
5. **Jellyfish gates** shift the bottleneck: fewer gates means less MSM work, but more complex SumChecks
6. For circuits where gate reduction exceeds SumCheck overhead → **zkPHIRE wins end-to-end**

### Follow-up Explorations
- Implement Lagrange interpolation using Barycentric form (hardware-efficient)
- Model the Multifunction Forest (binary tree for MLE evaluation / product MLE construction)
- Implement the Permutation Quotient Generator for PermCheck
- Explore design space: how does SumCheck PE count trade off vs MSM PEs?